In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import glob
import os

from microhhpy.spatial import calc_vertical_grid_2nd
from microhhpy.thermo import calc_moist_basestate
from microhhpy.chem import Emission_input

## Define MicroHH grid and base state.

See `grid_and_basestate.ipynb` for more details on the `gd` and `bs` dictionaries and their content.

In [ ]:
"""
Settings and MicroHH grid/fields.
"""
xsize = 300
ysize = 300
zsize = 300

itot = 300
jtot = 300
ktot = 300

dx = xsize / itot
dy = ysize / jtot
dz = zsize / ktot

x = np.arange(dx/2, xsize, dx)
y = np.arange(dy/2, ysize, dy)
z = np.arange(dz/2, zsize, dz)

# Calculate exact definition vertical grid.
# Not really necessary for equidistant grid, more important for e.g. stretched grids.
gd = calc_vertical_grid_2nd(z, zsize)

# Initial fields.
thl = 290 + 0.006 * z
qt  = np.zeros(ktot)
ps = 1e5

# Calculate base state model.
bs = calc_moist_basestate(thl, qt, ps, z, zsize)

In [ ]:
"""
Create emission input.
"""
times = np.array([0, 3600])     # Or simply np.array([0]) for non time-dependent emissions.

fields = ['s1']
emiss = Emission_input(fields, times, x, y, z, gd['dz'], bs['rho'])

emiss.add_gaussian(field='s1', strength=1, time=0, x0=150, y0=150, z0=100, sigma_x=50, sigma_y=25, sigma_z=25, sw_vmr=True)
emiss.add_gaussian(field='s1', strength=1, time=0, x0=50, y0=50, z0=50, sigma_x=10, sigma_y=10, sigma_z=10, sw_vmr=False)

emiss.add_gaussian(field='s1', strength=2, time=3600, x0=150, y0=150, z0=100, sigma_x=50, sigma_y=25, sigma_z=25, sw_vmr=True)
emiss.add_gaussian(field='s1', strength=2, time=3600, x0=50, y0=50, z0=50, sigma_x=10, sigma_y=10, sigma_z=10, sw_vmr=False)

# Clip vertical extent.
emiss.clip()

# Save as binary input for MicroHH.
# Resulting files are saved as `{path}/{name}_emission.{time:07d}`.
emiss.to_binary(path='.')

# Vertical extent emissions has to be provided in .ini as `source->ktot`, and is available as `emiss.kmax`.

In [ ]:
"""
Plot.
"""
plt.figure()
plt.pcolormesh(x, z[:emiss.kmax], emiss.data['s1'][0, :, :, :].sum(axis=1))
plt.colorbar()

In [ ]:
files = glob.glob('*00*')
for f in files:
    os.remove(f)